### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just use a linear model because it's 'simple'. You use it because it's **identifiable**. In a GLM, every parameter has a direct physical meaning (e.g., 'Increasing temperature by 1° increases claim frequency by 5%'). In production, interpretability is often more valuable than a 1% gain in accuracy.

**Mental Model:** 
- **Linear Regression:** A best-fit hyperplane through a cloud of data points.
- **Logistic Regression:** A linear model wrapped in a sigmoid — it models the *log-odds* of a binary outcome.
- **ElasticNet:** A 'smart' regularity filter that handles redundant features much better than Lasso alone.
- **GLMs:** A single mathematical machine that can be 'configured' to handle counts (Poisson), binary flips (Logistic), or ratings (Regression) by just swapping a 'Link Function'.

**Common Junior Pitfall:** Using MSE for count data (e.g., number of website clicks). Count data must be non-negative and follows a Poisson distribution. Using MSE assumes the noise can be negative, leading to physically impossible predictions (e.g., '-2 people visited the site today').

---


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Linear Regression — From Normal Equation to Gradient Descent](#1-linear-regression--from-normal-equation-to-gradient-descent)
  * [1.1 Loss Surface Visualization](#11-loss-surface-visualization)
* [2. Logistic Regression from Scratch](#2-logistic-regression-from-scratch)
  * [2.1 Decision Boundary Visualization](#21-decision-boundary-visualization)
* [3. Regularization — Ridge, Lasso, ElasticNet](#3-regularization--ridge-lasso-elasticnet)
  * [3.1 Regularization Path Visualization](#31-regularization-path-visualization)
  * [3.2 Coordinate Descent Implementation](#32-coordinate-descent-implementation)
* [4. Sklearn Pipeline — The Production Standard](#4-sklearn-pipeline--the-production-standard)
* [5. Generalized Linear Models (GLMs)](#5-generalized-linear-models-glms)
  * [5.1 Poisson Regression for Count Data](#51-poisson-regression-for-count-data)
  * [5.2 GLM Comparison Panel](#52-glm-comparison-panel)
* [6. Convergence Theory — The Mathematics of Speed](#6-convergence-theory--the-mathematics-of-speed)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)


## 1. Linear Regression — From Normal Equation to Gradient Descent

We aim to find $W$ that minimizes the loss surface $L = \frac{1}{2n}||Xw - y||^2$.

**Two approaches:**
- **Normal Equation:** $w^* = (X^TX)^{-1}X^Ty$ — Exact, $O(d^3)$, fails for large $d$
- **Gradient Descent:** Iterative, $O(nd)$ per step, scales to any size


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Real dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features (critical for GD convergence!)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Add intercept
X_tr_b = np.c_[np.ones(X_train_s.shape[0]), X_train_s]
X_te_b = np.c_[np.ones(X_test_s.shape[0]), X_test_s]

# Method 1: Normal Equation
w_normal = np.linalg.inv(X_tr_b.T @ X_tr_b) @ X_tr_b.T @ y_train

# Method 2: Gradient Descent
def gradient_descent(X, y, lr=0.01, n_iter=1000):
    m = len(y)
    w = np.random.randn(X.shape[1]) * 0.01
    losses = []
    for i in range(n_iter):
        grad = (2/m) * X.T @ (X @ w - y)
        w -= lr * grad
        losses.append(np.mean((X @ w - y)**2))
    return w, losses

w_gd, losses = gradient_descent(X_tr_b, y_train, lr=0.01, n_iter=500)

# Compare
y_pred_ne = X_te_b @ w_normal
y_pred_gd = X_te_b @ w_gd

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(losses, color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('MSE', fontsize=12)
axes[0].set_title('Gradient Descent Convergence', fontsize=14)

axes[1].scatter(y_test[:200], y_pred_ne[:200], alpha=0.5, s=20, color='#3498db', label=f'Normal Eq (R²={r2_score(y_test, y_pred_ne):.3f})')
axes[1].scatter(y_test[:200], y_pred_gd[:200], alpha=0.5, s=20, color='#e74c3c', label=f'GD (R²={r2_score(y_test, y_pred_gd):.3f})')
axes[1].plot([0, 5], [0, 5], 'k--', alpha=0.3)
axes[1].set_xlabel('Actual Price', fontsize=12)
axes[1].set_ylabel('Predicted Price', fontsize=12)
axes[1].set_title('Predicted vs Actual: California Housing', fontsize=14)
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Normal Equation MSE: {mean_squared_error(y_test, y_pred_ne):.4f}")
print(f"Gradient Descent MSE: {mean_squared_error(y_test, y_pred_gd):.4f}")

"""What just happened?
We trained linear regression on real California Housing data using both the exact
analytical solution and iterative gradient descent. Both converge to nearly identical
results. For 100 samples the Normal Equation is instant, but for massive datasets,
GD (and especially SGD) is the only feasible approach.
"""

### 1.1 Loss Surface Visualization

Linear regression has a **convex** loss surface — a single global minimum. This is why gradient descent always converges.


In [ ]:
# 2D loss surface for simple y = w0 + w1*x
np.random.seed(42)
x_simple = 2 * np.random.rand(50)
y_simple = 3 + 2 * x_simple + np.random.randn(50) * 0.3

w0_range = np.linspace(1, 5, 100)
w1_range = np.linspace(0, 4, 100)
W0, W1 = np.meshgrid(w0_range, w1_range)
Loss = np.array([[np.mean((y_simple - (w0 + w1 * x_simple))**2)
                  for w0, w1 in zip(w0_row, w1_row)]
                 for w0_row, w1_row in zip(W0, W1)])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 3D surface
ax3d = fig.add_subplot(121, projection='3d')
ax3d.plot_surface(W0, W1, Loss, cmap='viridis', alpha=0.8)
ax3d.set_xlabel('w0 (intercept)')
ax3d.set_ylabel('w1 (slope)')
ax3d.set_zlabel('MSE')
ax3d.set_title('3D Loss Surface (Convex Bowl)')

# Contour with GD trajectory
axes[1].contour(W0, W1, Loss, levels=30, cmap='viridis', alpha=0.7)

# Run GD and track path
w = np.array([1.0, 0.5])
lr = 0.05
path = [w.copy()]
X_s = np.c_[np.ones(50), x_simple]
for _ in range(50):
    grad = (2/50) * X_s.T @ (X_s @ w - y_simple)
    w -= lr * grad
    path.append(w.copy())
path = np.array(path)

axes[1].plot(path[:, 0], path[:, 1], 'ro-', markersize=3, linewidth=1.5)
axes[1].plot(path[0, 0], path[0, 1], 'r*', markersize=15, label='Start')
axes[1].plot(path[-1, 0], path[-1, 1], 'g*', markersize=15, label='End')
axes[1].set_xlabel('w0 (intercept)')
axes[1].set_ylabel('w1 (slope)')
axes[1].set_title('Contour + GD Trajectory')
axes[1].legend()
plt.tight_layout()
plt.show()

"""What just happened?
Left: the MSE loss surface is a perfect convex bowl — there is exactly one minimum.
Right: gradient descent follows the steepest downhill direction, spiraling toward
the optimum. The elliptical contours show that the surface is 'steeper' in the w1
direction — this is why feature scaling matters for convergence speed.
"""

## 2. Logistic Regression from Scratch

Logistic Regression models the **log-odds** of a binary outcome:
$$P(y=1|x) = \sigma(w^Tx) = \frac{1}{1 + e^{-w^Tx}}$$

**Loss function (Binary Cross-Entropy):**
$$L = -\frac{1}{n}\sum [y_i \log(\hat{p}_i) + (1-y_i)\log(1 - \hat{p}_i)]$$

This is derived from MLE assuming a Bernoulli distribution — see NB13_01.


In [ ]:
from sklearn.datasets import load_breast_cancer

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class LogisticRegressionScratch:
    def __init__(self, lr=0.1, n_iter=1000):
        self.lr, self.n_iter = lr, n_iter
        self.losses = []
        
    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        
        for _ in range(self.n_iter):
            z = X @ self.w + self.b
            p = sigmoid(z)
            
            # Binary cross-entropy
            loss = -np.mean(y * np.log(p + 1e-8) + (1-y) * np.log(1-p + 1e-8))
            self.losses.append(loss)
            
            # Gradients
            dw = (1/n) * X.T @ (p - y)
            db = (1/n) * np.sum(p - y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Train on Breast Cancer
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42, stratify=y_bc)

sc_bc = StandardScaler()
X_bc_tr_s = sc_bc.fit_transform(X_bc_tr)
X_bc_te_s = sc_bc.transform(X_bc_te)

lr_model = LogisticRegressionScratch(lr=0.1, n_iter=500)
lr_model.fit(X_bc_tr_s, y_bc_tr)

y_pred_lr = lr_model.predict(X_bc_te_s)
acc = np.mean(y_pred_lr == y_bc_te)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lr_model.losses, color='#e74c3c', linewidth=2)
ax.set_xlabel('Iteration');  ax.set_ylabel('Binary Cross-Entropy')
ax.set_title(f'Logistic Regression Convergence (Test Acc: {acc:.3f})', fontsize=14)
plt.tight_layout()
plt.show()

"""What just happened?
We built logistic regression from scratch using gradient descent on the binary
cross-entropy loss. The sigmoid squashes the linear output to [0,1], giving us
probabilities. Note how the loss decreases smoothly — logistic regression has a
convex loss surface, guaranteeing convergence to the global optimum.
"""

## 3. Regularization — Ridge, Lasso, ElasticNet

| Method | Penalty | Effect | Use Case |
|:---|:---|:---|:---|
| **Ridge (L2)** | $\lambda ||w||_2^2$ | Shrinks all weights | Correlated features |
| **Lasso (L1)** | $\lambda ||w||_1$ | Sets some weights to exactly 0 | Feature selection |
| **ElasticNet** | $\lambda_1 ||w||_1 + \lambda_2 ||w||_2^2$ | Both | Best of both worlds |

**Why ElasticNet?** If two features are 99% correlated, Lasso arbitrarily keeps one and drops the other. ElasticNet keeps both but shares the weight — the **grouping effect**.


### 3.1 Regularization Path Visualization

The **regularization path** shows how each coefficient changes as $\lambda$ increases. Lasso drives coefficients to *exact zero*, while Ridge only shrinks them.


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

alphas = np.logspace(-3, 3, 100)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge = Ridge(alpha=a).fit(X_bc_tr_s, y_bc_tr)
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_bc_tr_s, y_bc_tr)
    ridge_coefs.append(ridge.coef_)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i in range(min(10, ridge_coefs.shape[1])):
    axes[0].plot(alphas, ridge_coefs[:, i], linewidth=1.5)
    axes[1].plot(alphas, lasso_coefs[:, i], linewidth=1.5)

for ax, title in zip(axes, ['Ridge (L2): Coefficients Shrink', 'Lasso (L1): Coefficients Hit Zero']):
    ax.set_xscale('log')
    ax.set_xlabel('Alpha (Regularization Strength)', fontsize=12)
    ax.set_ylabel('Coefficient Value', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.axhline(0, color='black', linewidth=0.5)

plt.suptitle('Regularization Paths: How Coefficients Respond to Penalty', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Count zero coefficients at strong regularization
strong_lasso = Lasso(alpha=0.1, max_iter=5000).fit(X_bc_tr_s, y_bc_tr)
print(f"\nLasso at alpha=0.1: {np.sum(strong_lasso.coef_ == 0)}/{len(strong_lasso.coef_)} coefficients are EXACTLY zero")
print("These features have been 'selected out' — automatic feature selection!")

"""What just happened?
Left: Ridge smoothly shrinks all coefficients toward zero but never reaches it.
Right: Lasso aggressively drives coefficients to EXACT zero, performing automatic
feature selection. This is the geometric reason: L1's diamond-shaped constraint
region has corners on the axes, where solutions naturally land.
"""

In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0)

def elastic_net_cd(X, y, l1=1.0, l2=1.0, n_iter=100):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    losses = []
    
    for it in range(n_iter):
        for j in range(n_features):
            r_j = y - (X @ w - X[:, j] * w[j])
            rho_j = X[:, j] @ r_j
            denominator = (X[:, j] @ X[:, j]) + l2
            w[j] = soft_threshold(rho_j, l1) / denominator
        losses.append(np.mean((y - X @ w)**2))
    return w, losses

# Create correlated features to show grouping effect
np.random.seed(42)
n = 200
x1 = np.random.randn(n)
x2 = x1 + np.random.randn(n) * 0.05  # 99% correlated with x1
x3 = np.random.randn(n)               # Independent
X_corr = np.column_stack([x1, x2, x3])
y_corr = 3*x1 + 3*x2 + 0.5*x3 + np.random.randn(n)*0.1

# Compare Lasso vs ElasticNet
w_lasso, _ = elastic_net_cd(X_corr, y_corr, l1=0.5, l2=0.0)
w_en, _ = elastic_net_cd(X_corr, y_corr, l1=0.5, l2=0.5)

print(f"Lasso weights:      {w_lasso} (one correlated feature killed!)")
print(f"ElasticNet weights: {w_en} (both correlated features preserved!)")
print(f"\nGrouping Effect: ElasticNet gives similar weights to correlated X1 and X2.")

"""What just happened?
We implemented Coordinate Descent with Soft Thresholding. Lasso arbitrarily kills
one of two correlated features. ElasticNet preserves both with shared weight — the
'grouping effect'. In genomics or NLP where features are naturally correlated,
ElasticNet is far more stable than pure Lasso.
"""

## 4. Sklearn Pipeline — The Production Standard

In production, you NEVER call `scaler.fit_transform()` and `model.fit()` separately. This causes **data leakage** when scaling uses test data statistics.

A `Pipeline` ensures that preprocessing is fitted ONLY on training data during cross-validation.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import cross_val_score

# Full production pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('model', ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 0.99], cv=5, max_iter=5000))
])

# Cross-validated evaluation — NO data leakage!
X_cal, y_cal = fetch_california_housing(return_X_y=True)
scores = cross_val_score(pipe, X_cal, y_cal, cv=5, scoring='r2')

pipe.fit(X_train, y_train)
best_model = pipe.named_steps['model']

print(f"Cross-validated R²: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"Best l1_ratio:      {best_model.l1_ratio_:.2f}")
print(f"Best alpha:         {best_model.alpha_:.6f}")
print(f"Non-zero features:  {np.sum(best_model.coef_ != 0)}/{len(best_model.coef_)}")
print(f"\nThe pipeline guarantees NO data leakage: scaling is fit only on train folds.")

"""What just happened?
We built a production-grade pipeline: StandardScaler -> PolynomialFeatures -> 
ElasticNetCV. The CV in ElasticNetCV automatically selects the best alpha and
l1_ratio via cross-validation. The outer cross_val_score gives an unbiased estimate.
The pipeline ensures scaling is NEVER fit on test data.
"""

## 5. Generalized Linear Models (GLMs)

A GLM is a unifying framework for all regression. It consists of:
1. **Random Component:** The distribution family (Gaussian, Bernoulli, Poisson, Gamma).
2. **Systematic Component:** The linear predictor $\eta = Xw$.
3. **Link Function:** Maps $\eta$ to the predicted mean.

| Distribution | Link Function | Use Case |
|:---|:---|:---|
| Gaussian | Identity ($\mu = \eta$) | Continuous data (house prices) |
| Bernoulli | Logit ($\log \frac{p}{1-p} = \eta$) | Binary outcomes (click/no-click) |
| Poisson | Log ($\log \lambda = \eta$) | Count data (website visits) |
| Gamma | Inverse ($\frac{1}{\mu} = \eta$) | Positive continuous (insurance claims) |


In [ ]:
from sklearn.linear_model import PoissonRegressor

# Simulate count data: insurance claims based on risk factors
np.random.seed(42)
n = 500
age = np.random.uniform(18, 65, n)
risk_score = np.random.uniform(0, 5, n)
X_claims = np.column_stack([age / 65, risk_score / 5])  # Normalize

# True model: lambda = exp(0.5 + 0.8*risk + 0.3*age)
true_lambda = np.exp(0.5 + 0.8 * X_claims[:, 1] + 0.3 * X_claims[:, 0])
y_claims = np.random.poisson(true_lambda)

# Fit Poisson regression
poisson_model = PoissonRegressor(alpha=0.01, max_iter=1000)
poisson_model.fit(X_claims, y_claims)

# Compare Poisson vs OLS
from sklearn.linear_model import LinearRegression
ols_model = LinearRegression().fit(X_claims, y_claims)

X_plot = np.column_stack([np.full(100, 0.5), np.linspace(0, 1, 100)])
pred_poisson = poisson_model.predict(X_plot)
pred_ols = ols_model.predict(X_plot)

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_claims[:, 1], y_claims, alpha=0.3, s=15, color='gray', label='Observed Claims')
ax.plot(X_plot[:, 1], pred_poisson, color='#e74c3c', linewidth=2.5, label='Poisson GLM')
ax.plot(X_plot[:, 1], pred_ols, color='#3498db', linewidth=2.5, linestyle='--', label='OLS (Linear)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Risk Score (normalized)', fontsize=12)
ax.set_ylabel('Number of Claims', fontsize=12)
ax.set_title('Poisson GLM vs OLS: Count Data Requires the Right Model', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"OLS predicts negative claims for low risk: {pred_ols[0]:.2f} (IMPOSSIBLE!)")
print(f"Poisson always predicts non-negative:      {pred_poisson[0]:.2f} (Correct!)")

"""What just happened?
OLS produces negative predictions for low-risk customers — physically impossible for
count data. The Poisson GLM uses a log-link ensuring predictions are always positive.
The exponential curve also correctly captures the non-linear relationship between
risk and claim frequency.
"""

## 6. Convergence Theory — The Mathematics of Speed

**L-Lipschitz Gradients:** If the gradient doesn't change faster than some constant $L$, then with $lr = 1/L$, GD is guaranteed to converge at rate $O(1/t)$.

**Strong Convexity ($\mu$):** If the loss surface is 'pointy' enough, we get **Exponential Convergence**:
$$||w_t - w^*|| \le \left(1 - \frac{\mu}{L}\right)^t ||w_0 - w^*||$$

**Condition Number** $\kappa = L/\mu$: The ratio of the flattest to steepest direction. High $\kappa$ means slow convergence in the flattest direction.

🎓 **Socratic Deep Check:**
> *"If you add massive L2 Regularization to your model, you are physically INCREASING the strong convexity constant $\mu$. Why does this make your model converge MUCH faster, even though it might decrease your peak accuracy?"*


In [ ]:
L, mu = 100.0, 1.0
iterations = np.arange(100)

# Compare convergence rates
loss_gd = (1 - mu/L)**iterations           # Standard GD
loss_reg = (1 - (mu+10)/(L+10))**iterations  # With L2 reg (mu+10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iterations, np.log10(loss_gd), label=f'No Regularization (kappa={L/mu:.0f})', 
        color='#e74c3c', linewidth=2)
ax.plot(iterations, np.log10(loss_reg), label=f'With L2 (kappa={(L+10)/(mu+10):.1f})',
        color='#3498db', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Log10(Error)', fontsize=12)
ax.set_title('Convergence Speed: Regularization Improves Condition Number', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Without regularization: kappa = {L/mu:.0f} (slow convergence)")
print(f"With L2 regularization: kappa = {(L+10)/(mu+10):.1f} (fast convergence)")

"""What just happened?
L2 regularization physically reshapes the loss surface, making it more 'round' (lower
condition number). This trades off a small amount of accuracy for dramatically faster
convergence. In production with tight latency budgets, this tradeoff is often worth it.
"""

---
## ✅ Knowledge Check

### Q1: Why use GLM for Insurance?
<details><summary>👉 Answer</summary>
Insurance features are often multiplicative ('Adding a car alarm reduces theft risk by 20%'). GLMs with a log-link (Poisson) automatically transform additive linear weights into multiplicative factors ($\exp(w) \cdot \exp(x)$), which perfectly matches the business logic.
</details>

### Q2: ElasticNet vs Lasso/Ridge
<details><summary>👉 Answer</summary>
Lasso kills features aggressively; Ridge shrinks them evenly. ElasticNet provides Lasso's sparsity with Ridge's stability and grouping effect for correlated features.
</details>

### Q3: Why does scaling matter for GD?
<details><summary>👉 Answer</summary>
Without scaling, the loss surface is elongated (high condition number). GD oscillates in the steep direction and crawls in the flat direction. Scaling makes the surface circular, allowing equal progress in all directions with a single learning rate.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement the sigmoid function and verify that $\sigma(0) = 0.5$, $\sigma(\infty) = 1$, $\sigma(-\infty) = 0$.
2. Modify the OLS `gradient_descent` to use **SGD** (Stochastic GD) with batch_size=32. Compare convergence speed.

### Tier 2: Intermediate
1. **Multi-Collinearity Stress Test:** Create a dataset where $X_1$ and $X_2$ are 99% correlated. Train Lasso, Ridge, and ElasticNet. Show that only ElasticNet preserves both features stably.
2. **Pipeline + GridSearchCV:** Build a pipeline with StandardScaler + PolynomialFeatures(degree=[1,2,3]) + Ridge. Use GridSearchCV to find the best degree and alpha jointly.

### Tier 3: Advanced
1. **Gamma Regression from Scratch:** Using the GLM framework, implement Gamma Regression (inverse link) for predicting strictly positive, long-tailed data like 'Time to Failure' in manufacturing.
2. **IRLS (Iteratively Reweighted Least Squares):** Implement the IRLS algorithm used to fit GLMs. Show that it converges faster than plain gradient descent on Poisson regression.
